# Experiment 013: does the original 7B configuration fit an 80 GB A100? (Pilot A) (Colab)

30 grpo training steps at the exact execution settings every other scale used
(per-device 8, accum 1, no checkpointing), on the raw 7b base. Training only,
the test set is never evaluated, and the oom ladder is off, so this either
completes at the declared config or fails. The verdict comes from the
pre-registered rule, not from judgement:

PASS = completes 30 steps AND peak_reserved / total_memory <= 0.95

Pass -> all 7b runs use the original settings and the only difference left vs
3b is the model itself. Fail -> pilot B (4x2 + checkpointing) in its own
notebook. Estimated ~30min. Select the a100 80gb runtime before running.

## 0. Compute units and mount Drive

In [9]:
# note the balance before anything burns units, then save it after drive mounts
import glob, json, os, time
session_id = globals().get('session_id', time.strftime('%Y%m%d_%H%M%S'))
cu_before = 206.23  # e.g. 87.3
print('session', session_id, '| cu_before =', cu_before if cu_before is not None else 'FILL ME')

from google.colab import drive
drive.mount('/content/drive')
ROOT = '/content/drive/MyDrive/inm363'
RESULTS = os.path.join(ROOT, 'results')
OPEN_UNITS = os.path.join(RESULTS, 'open_cu_sessions')
UNITS_LOG = os.path.join(RESULTS, 'colab_units.jsonl')
os.makedirs(OPEN_UNITS, exist_ok=True)

# close unfinished session with the balance captured at this runtime start
old_sessions = [p for p in glob.glob(os.path.join(OPEN_UNITS, '*.json'))
                if not p.endswith(f'{session_id}.json')]
open_path = os.path.join(OPEN_UNITS, f'{session_id}.json')
if cu_before is None:
    print('reminder: fill cu_before, then rerun this cell to save the session')
else:
    for old_path in old_sessions:
        with open(old_path) as f:
            old = json.load(f)
        old['ts_end'] = time.strftime('%Y-%m-%d %H:%M')
        old['colab_cu_after'] = cu_before
        old['colab_cu_used'] = round(old['colab_cu_before'] - cu_before, 2)
        old['closed_on_resume'] = True
        with open(UNITS_LOG, 'a') as f:
            f.write(json.dumps(old) + '\n')
        os.remove(old_path)
        print('closed previous session:', old['session_id'])

    if not os.path.exists(open_path):
        start = {'session_id': session_id, 'ts_start': time.strftime('%Y-%m-%d %H:%M'),
                 'notebook': '013_pilot_a', 'colab_cu_before': cu_before}
        with open(open_path, 'w') as f:
            json.dump(start, f, indent=2)
        print('saved open session:', session_id)
print('drive mounted ->', ROOT)


session 20260722_122507 | cu_before = 206.23
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
drive mounted -> /content/drive/MyDrive/inm363


## 1. GPU check: A100 with 80GB required

In [10]:
import subprocess, torch
gpu = subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True).stdout
print(gpu.strip() or 'no gpu visible')
total_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f'total vram: {total_gb:.1f} GiB')
assert 'A100' in gpu and total_gb >= 79, 'need the a100 80gb runtime for the 7b pilot'

GPU 0: NVIDIA A100-SXM4-80GB (UUID: GPU-43f8a244-050f-8fc2-daf2-e25983b54875)
total vram: 79.3 GiB


## 2. Clone or update the repo (automatic)

In [11]:
import base64
import getpass
import glob
import os
from pathlib import Path
import subprocess

REPO_SLUG = 'hastov-lab/rl-grpo-or-sft-for-llms'
repo_url = f'https://github.com/{REPO_SLUG}.git'
repo_root = Path('/content') / REPO_SLUG.split('/')[-1]


def repo_git(*args):
    git = ['git']
    env = None
    try:
        from google.colab import userdata
    except ImportError:
        userdata = None

    # Private forks can use a read token from Colab Secrets.
    if userdata is not None and REPO_SLUG != 'hastov-lab/rl-grpo-or-sft-for-llms':
        try:
            token = userdata.get('GH_TOKEN')
        except Exception:
            token = None
        if isinstance(token, dict):
            token = token.get('value')
        if not isinstance(token, str) or not token.strip():
            token = getpass.getpass('GitHub token: ')
        if not token.strip():
            raise RuntimeError('This private fork requires a GitHub token with read access.')

        # Keep the token out of the command line and the saved remote URL.
        auth = base64.b64encode(f'x-access-token:{token.strip()}'.encode()).decode()
        env = dict(os.environ, REPO_GIT_AUTH=f'Authorization: Basic {auth}')
        git.append('--config-env=http.https://github.com/.extraHeader=REPO_GIT_AUTH')

    subprocess.run([*git, *args], env=env, check=True)


if repo_root.exists():
    if not (repo_root / '.git').exists():
        raise RuntimeError(f'Expected a Git checkout at {repo_root}.')
    origin = subprocess.check_output(
        ['git', '-C', str(repo_root), 'config', '--local', '--get', 'remote.origin.url'], text=True
    ).strip()
    if origin != repo_url:
        raise RuntimeError(f'The checkout at {repo_root} belongs to a different repository.')
    # a resumed runtime already has the clone, in that case pull instead of failing
    # and silently running whatever old code is lying around
    branch = subprocess.check_output(
        ['git', '-C', str(repo_root), 'branch', '--show-current'], text=True
    ).strip()
    if branch != 'main':
        raise RuntimeError(f'Expected the main branch at {repo_root}.')
    repo_git('-C', str(repo_root), 'pull', '--ff-only', 'origin', 'main')
else:
    repo_git('clone', '--branch', 'main', '--single-branch', '--depth', '1', repo_url, str(repo_root))

# Use the scripts from this checkout even if another experiment is open.
code_root = repo_root / '001_Code'
if not (code_root / 'scripts' / 'run.py').is_file():
    raise RuntimeError(f'Experiment scripts are missing from {code_root}.')
os.chdir(code_root)
print('working in:', os.getcwd())

Already up to date.
working in: /content/INM363-IndividualProject-Dissertation/001_Code


## 3. Install dependencies

In [12]:
!pip install -q trl==1.7.1 transformers==5.5.4 peft==0.19.1 accelerate==1.14.0 datasets==4.8.4 wandb pyyaml
# Colab preinstalls a torchao that peft 0.19.1 chokes on inside its lora
# dispatcher. No quantization here, so just drop it
!pip uninstall -q -y torchao
print('deps installed')


deps installed


## 4. Drive space and unit rate

In [13]:
import shutil
free_gb = shutil.disk_usage('/content/drive/MyDrive').free / 1024**3
print(f'drive free: {free_gb:.0f} GB')
assert free_gb >= 60, 'need >=60GB free on drive before the 7b phases'
# the pilot exists partly to measure cost, so the rate is required here, not
# optional like in the long unattended runs
unit_rate = 6.77    # from the resources panel, e.g. 6.77
print('unit rate this session:', unit_rate if unit_rate is not None else 'FILL ME (required)')

drive free: 179 GB
unit rate this session: 6.77


## 5. Build the data splits

In [14]:
!python scripts/build_data.py


dropped 1 grpo prompt(s) that also sit in the test split:
  [high_school_psychology] 'Answer the multiple choice question. Think through it step by step inside <think></think> tags, then'...
sft rows:  7473
grpo rows: 245  by tier: {'easy': 110, 'medium': 82, 'hard': 53}
eval rows: 1884  by tier: {'easy': 879, 'medium': 609, 'hard': 396}
no leakage, three splits ready


## 6. The pilot run

Direct call into run_grpo, no runner, no evals, isolated pilot dirs. The
report json lands on drive either way, pass or fail.

In [15]:
import gc, json, os, sys, time, traceback
from datetime import datetime
sys.path.insert(0, os.getcwd())
import torch
from config import load_config
from scripts.env_switch import resolve_device, apply_smoke
from train.grpo import run_grpo
from train.model_utils import resolve_dtype
from obs import projected_hours, gpu_snapshot, _git_info, _versions

# both cost fields must be filled before any gpu time burns, this session is
# the cost measurement
assert cu_before is not None, 'fill cu_before in the units cell first'
assert unit_rate is not None, 'fill unit_rate in the drive/rate cell first'

cfg = apply_smoke(load_config('configs/013_pilot_a.yaml'))
device = resolve_device(cfg.env)
D = cfg.output_root
out_dir = os.path.join(D, 'checkpoints', '013_pilot_a', 'grpo')
report_dir = os.path.join(D, 'results', '013_pilot_a')
os.makedirs(report_dir, exist_ok=True)

# Attempt A is pre-registered and runs once. A second run would overwrite the
# first measurement, so refuse instead
report_path = os.path.join(report_dir, 'pilot_report.json')
assert not os.path.exists(report_path), f'pilot A already ran, see {report_path}'

# clean slate so the recorded peak belongs to this attempt alone
gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

t0 = time.time()
completed = False
error = None
try:
    run_grpo(cfg, device, adapter_path=None, out_dir=out_dir, report_to=[],
             log_dir=report_dir, run_name='013_pilot_a', allow_oom_fallback=False)
    completed = True
except Exception as e:
    error = f'{type(e).__name__}: {e}'
    traceback.print_exc()
wall = time.time() - t0

# Projection from the trainer's own per-step times, one-time load/save
# overheads excluded. Wall stays in the report for reference only
step_times, train_runtime, steps_done = [], None, 0
tl_path = os.path.join(report_dir, 'grpo_trainlog.json')
if os.path.exists(tl_path):
    tl = json.load(open(tl_path))
    step_times = [e['step_time'] for e in tl if 'step_time' in e]
    steps_done = max((e.get('step', 0) for e in tl), default=0)
    train_runtime = next((e['train_runtime'] for e in tl if 'train_runtime' in e), None)

total = torch.cuda.get_device_properties(0).total_memory
alloc = torch.cuda.max_memory_allocated()
reserved = torch.cuda.max_memory_reserved()
ratio = reserved / total
passed = completed and steps_done == 30 and ratio <= 0.95
proj_hours = round(projected_hours(step_times, 500), 2) if step_times else None

report = {'attempt': 'A', 'timestamp': datetime.now().isoformat(timespec='seconds'),
          'exec': {'grpo_batch_size': 8, 'grpo_grad_accum': 1, 'gradient_checkpointing': False},
          'completed': completed, 'error': error, 'steps_done': steps_done,
          'wall_sec': round(wall, 1), 'train_runtime_sec': train_runtime,
          'projected_500_step_hours': proj_hours,
          'total_memory_gib': round(total / 1024**3, 2),
          'max_memory_allocated_gib': round(alloc / 1024**3, 2),
          'max_memory_reserved_gib': round(reserved / 1024**3, 2),
          'reserved_over_total': round(ratio, 4), 'pass_rule': 'completed all 30 steps and reserved/total <= 0.95',
          'passed': passed, 'unit_rate': unit_rate, 'cu_before': cu_before,
          'gpu': gpu_snapshot(), 'resolved_dtype': resolve_dtype(cfg),
          'git': _git_info(), 'versions': _versions()}
with open(report_path, 'w') as f:
    json.dump(report, f, indent=2)

print(json.dumps({k: v for k, v in report.items() if k not in ('git', 'versions')}, indent=2))
print()
print('VERDICT:', 'PASS, adopt 8x1 for all 7b runs' if passed else 'FAIL, pilot B (4x2 + checkpointing) is next')

Output hidden; open in https://colab.research.google.com to view.

## 7. Close the units record

In [17]:
# closing balance, completes the open record for this session
import json, os, time
cu_after = 202.78    # fill from the resources panel again
if cu_after is None:
    print('reminder: fill cu_after for the record')
elif not os.path.isdir('/content/drive/MyDrive'):
    print('drive not mounted, run the mount cell first')
else:
    os.makedirs(OPEN_UNITS, exist_ok=True)
    open_path = os.path.join(OPEN_UNITS, f'{session_id}.json')
    if not os.path.exists(open_path):
        print('no open record for this session, it may already be closed')
    else:
        with open(open_path) as f:
            rec = json.load(f)
        rec['ts_end'] = time.strftime('%Y-%m-%d %H:%M')
        rec['colab_cu_after'] = cu_after
        rec['colab_cu_used'] = round(rec['colab_cu_before'] - cu_after, 2)
        rec['closed_on_resume'] = False
        with open(UNITS_LOG, 'a') as f:
            f.write(json.dumps(rec) + '\n')
        os.remove(open_path)
        print('logged:', rec)

logged: {'session_id': '20260722_122507', 'ts_start': '2026-07-22 12:25', 'notebook': '013_pilot_a', 'colab_cu_before': 206.23, 'ts_end': '2026-07-22 12:58', 'colab_cu_after': 202.78, 'colab_cu_used': 3.45, 'closed_on_resume': False}
